In [ ]:
# 实现一个简单的ListADT
class MyListADT:
    def __init__(self):
        # code
    def insert(self,value): # 末尾插入
        pass
    def delete(self,index): # 删除指定index的元素
        pass
    def get(self,index): # 返回指定index处的元素
        pass
    def size(self): # 返回当前长度
        pass

# ex1
class MyListADT:
    def __init__(self):
        self._data = [] # 空list

    def size(self):
        return len(self._data)



In [1]:
# Quick test: run notebook code in order
import random

class Stack:
    def __init__(self):
        self._items = []
    def push(self, item): self._items.append(item)
    def pop(self): return self._items.pop() if self._items else None
    def isEmpty(self): return len(self._items) == 0

class Cell:
    def __init__(self, row_index, col_index):
        self.row_index, self.col_index = row_index, col_index
        self.N = self.S = self.E = self.W = True
    def coordinate(self): return (self.row_index, self.col_index)
    def breakWall(self, direction):
        if direction == "N": self.N = False
        elif direction == "S": self.S = False
        elif direction == "E": self.E = False
        elif direction == "W": self.W = False

def getNeighbors(maze, cell, width, height):
    r, c = cell.coordinate()
    neighbors = []
    if r > 0: neighbors.append((maze[r-1][c], "N"))
    if r < height-1: neighbors.append((maze[r+1][c], "S"))
    if c > 0: neighbors.append((maze[r][c-1], "W"))
    if c < width-1: neighbors.append((maze[r][c+1], "E"))
    return neighbors

def breakWallBetweenTwoCells(u, v):
    ur, uc = u.coordinate()
    vr, vc = v.coordinate()
    if vr == ur-1 and vc == uc: u.breakWall("N"); v.breakWall("S")
    elif vr == ur+1 and vc == uc: u.breakWall("S"); v.breakWall("N")
    elif vr == ur and vc == uc-1: u.breakWall("W"); v.breakWall("E")
    elif vr == ur and vc == uc+1: u.breakWall("E"); v.breakWall("W")

def draw_maze(maze, width, height):
    for r in range(height):
        line_top = "+"
        for c in range(width):
            line_top += "--" if maze[r][c].N else "  "
            line_top += "+"
        print(line_top)
        line_mid = ""
        for c in range(width):
            line_mid += "|" if maze[r][c].W else " "
            line_mid += "  "
        line_mid += "|" if maze[r][width - 1].E else " "
        print(line_mid)
    line_bot = "+"
    for c in range(width):
        line_bot += "--" if maze[height - 1][c].S else "  "
        line_bot += "+"
    print(line_bot)

def maze_generation(width, height):
    maze = [[Cell(r, c) for c in range(width)] for r in range(height)]
    visited = [[False] * width for _ in range(height)]
    stack = Stack()
    stack.push(maze[0][0])
    visited[0][0] = True
    while not stack.isEmpty():
        current = stack.pop()
        unvisited = [(nb, d) for nb, d in getNeighbors(maze, current, width, height) if not visited[nb.row_index][nb.col_index]]
        if unvisited:
            stack.push(current)
            next_cell, direction = random.choice(unvisited)
            breakWallBetweenTwoCells(current, next_cell)
            visited[next_cell.row_index][next_cell.col_index] = True
            stack.push(next_cell)
    return maze

class Node:
    def __init__(self, coord): self.coord = coord; self.N = self.S = self.E = self.W = None
    def get_next_and_direction(self):
        if self.N: return (self.N, "N")
        if self.S: return (self.S, "S")
        if self.E: return (self.E, "E")
        if self.W: return (self.W, "W")
        return (None, None)

class LinkedList:
    def __init__(self): self.head = None
    def get_tail(self):
        if self.head is None: return None
        node = self.head
        while True:
            n, _ = node.get_next_and_direction()
            if n is None: return node
            node = n
    def get_parent_of_tail(self):
        if self.head is None: return None, None
        next_n, _ = self.head.get_next_and_direction()
        if next_n is None: return None, None
        prev, node = self.head, next_n
        while True:
            next_n, _ = node.get_next_and_direction()
            if next_n is None:
                if prev.N is node: return prev, "N"
                if prev.S is node: return prev, "S"
                if prev.E is node: return prev, "E"
                if prev.W is node: return prev, "W"
                return prev, None
            prev, node = node, next_n
    def __str__(self):
        if self.head is None: return ""
        parts = []; node = self.head
        r, c = node.coord; parts.append(f"({r}, {c})")
        while True:
            next_n, direction = node.get_next_and_direction()
            if next_n is None: break
            parts.append(f"+-{direction}-+"); r, c = next_n.coord; parts.append(f"({r}, {c})"); node = next_n
        return "".join(parts)
    def add(self, newNode):
        if self.head is None: self.head = newNode; return
        tail = self.get_tail(); tr, tc = tail.coord; nr, nc = newNode.coord
        dr, dc = nr - tr, nc - tc
        if dr == -1 and dc == 0: tail.N = newNode
        elif dr == 1 and dc == 0: tail.S = newNode
        elif dr == 0 and dc == -1: tail.W = newNode
        elif dr == 0 and dc == 1: tail.E = newNode
    def removeLastNode(self):
        if self.head is None: return
        if self.head.get_next_and_direction()[0] is None: self.head = None; return
        parent, direction = self.get_parent_of_tail()
        if direction == "N": parent.N = None
        elif direction == "S": parent.S = None
        elif direction == "E": parent.E = None
        elif direction == "W": parent.W = None

def maze_resolution(maze):
    height, width = len(maze), len(maze[0])
    end = (height - 1, width - 1)
    path = LinkedList(); path.add(Node((0, 0)))
    visited = {(0, 0)}
    while True:
        tail = path.get_tail()
        if tail is None: return ""
        if tail.coord == end: return path.__str__()
        r, c = tail.coord
        candidates = []
        if r > 0 and not maze[r][c].N and (r-1, c) not in visited: candidates.append((r-1, c))
        if r < height-1 and not maze[r][c].S and (r+1, c) not in visited: candidates.append((r+1, c))
        if c > 0 and not maze[r][c].W and (r, c-1) not in visited: candidates.append((r, c-1))
        if c < width-1 and not maze[r][c].E and (r, c+1) not in visited: candidates.append((r, c+1))
        if not candidates:
            visited.discard(tail.coord)
            path.removeLastNode()
            continue
        next_coord = candidates[0]
        visited.add(next_coord)
        path.add(Node(next_coord))

if __name__ == "__main__":
    print("--- draw_maze 2x2 ---")
    w, h = 2, 2
    maze = [[Cell(r, c) for c in range(w)] for r in range(h)]
    maze[0][0].breakWall("E"); maze[0][1].breakWall("W")
    draw_maze(maze, w, h)
    print("--- maze_generation 4x4 + draw_maze ---")
    maze2 = maze_generation(4, 4)
    draw_maze(maze2, 4, 4)
    sol = maze_resolution(maze2)
    print("--- solution ---")
    print(sol)
    print("Format ok:", sol.startswith("(0, 0)") and "+-" in sol)


--- draw_maze 2x2 ---
+--+--+
|     |
+--+--+
|  |  |
+--+--+
--- maze_generation 4x4 + draw_maze ---
+--+--+--+--+
|  |        |
+  +  +--+  +
|     |     |
+--+--+  +--+
|        |  |
+  +--+--+  +
|           |
+--+--+--+--+
--- solution ---
(0, 0)+-S-+(1, 0)+-E-+(1, 1)+-N-+(0, 1)+-E-+(0, 2)+-E-+(0, 3)+-S-+(1, 3)+-W-+(1, 2)+-S-+(2, 2)+-W-+(2, 1)+-W-+(2, 0)+-S-+(3, 0)+-E-+(3, 1)+-E-+(3, 2)+-E-+(3, 3)
Format ok: True


In [ ]:
# lt1
class Octopus:
    def __init__(self):
        self._value = 0.0 # private: float value
        self.p = None # Person p (initialize to None or a Person instance)
        self.n = None # Credit_Card_No (Initialize to None or a instance)
    def Topup_value(self, amount):
    def Pay_Transaction(self, amount):
    def displayAmount(self, amount):

emptyList = list()
emptyList = []
myList = [1, 2, 3, 4, 5]
# heterogenous
mixedList = ["DSAI2201", 2201, 22.01, ["Data Structure"]]


In [2]:
myList = list(range(10))
print(myList)
print(myList[-1:0:-2])

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[9, 7, 5, 3, 1]


In [5]:
>>> stuff = list()
>>> stuff.append('book')
>>> stuff.append(99)
>>> print(stuff)
['book', 99]
>>> stuff.append('cookie')
>>> print(stuff)
['book', 99, 'cookie']

['book', 99]
['book', 99, 'cookie']


['book', 99, 'cookie']

In [13]:
t = ['a', 'b', 'c']
x = t.pop(1) # 如果不写就是pop最后一个
print(t)

t = ['a', 'b', 'c']
del t[1]
print(t)

t = ['a', 'b', 'b', 'c']
t.remove('b')
print(t)

t = ['a', 'b', 'c', 'd', 'e',  'f']
del t[1:5]
print(t)

['a', 'c']
['a', 'c']
['a', 'b', 'c']
['a', 'f']
